In [ ]:
# Healthcare dataset — load & prep for modeling
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt


In [ ]:
# Dataset: Kaggle healthcare (downloads if needed)
import kagglehub

path = kagglehub.dataset_download("prasad22/healthcare-dataset")
root = Path(path)
print("Data folder:", root)
print("Files:", [p.name for p in sorted(root.iterdir())])


In [ ]:
df = pd.read_csv(root / "healthcare_dataset.csv")
df["Name"] = df["Name"].str.title()
print(df.shape)
df.head()


In [ ]:
df.info()
df.isnull().sum()
print("Duplicates:", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates().copy()
print("After dedupe:", df.shape)


In [ ]:
# EDA (uses Name — before modeling drops it)
df_above_50 = df[df["Age"] > 50].sort_values("Age", ascending=True)
df_above_50["Medical Condition"].unique()


In [ ]:
df_above_50.groupby("Medical Condition")["Name"].count().sort_values(ascending=True)


In [ ]:
multi_disease = df_above_50.groupby("Name")["Medical Condition"].nunique()
multi_disease[multi_disease > 1]


In [ ]:
df["Date of Admission"] = pd.to_datetime(df["Date of Admission"])
df["Discharge Date"] = pd.to_datetime(df["Discharge Date"])


In [ ]:
# Modeling table: drop high-cardinality identifiers
df_model = df.drop(columns=["Name", "Doctor", "Hospital"]).copy()


In [ ]:
# Engineered features, then drop raw dates
df_model["length_of_stay"] = (df_model["Discharge Date"] - df_model["Date of Admission"]).dt.days
df_model["admission_year"] = df_model["Date of Admission"].dt.year
df_model["admission_month"] = df_model["Date of Admission"].dt.month
df_model = df_model.drop(columns=["Date of Admission", "Discharge Date"])


In [ ]:
cat_cols = [
    "Gender",
    "Blood Type",
    "Medical Condition",
    "Insurance Provider",
    "Medication",
    "Admission Type",
    "Test Results",
]

df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)


In [ ]:
# sklearn needs numeric columns only
df_encoded = df_encoded.replace({True: 1, False: 0})

if "Room Number" in df_encoded.columns:
    df_encoded = df_encoded.drop(columns=["Room Number"])

TARGET = "Billing Amount"
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

non_num = X.select_dtypes(include=["object", "boolean"]).columns.tolist()
assert len(non_num) == 0, f"Non-numeric columns remain: {non_num}"
X.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Scale only continuous columns; keep one-hot columns as 0/1
base_numeric = ["Age", "length_of_stay", "admission_year", "admission_month"]
numeric_features = [c for c in base_numeric if c in X.columns]
dummy_features = [c for c in X.columns if c not in numeric_features]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("pass", "passthrough", dummy_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lin_pipe = Pipeline([("prep", preprocess), ("model", LinearRegression())])
lin_pipe.fit(X_train, y_train)
y_pred_lin = lin_pipe.predict(X_test)

print("LinearRegression")
print("  R2:", r2_score(y_test, y_pred_lin))
print("  MAE:", mean_absolute_error(y_test, y_pred_lin))
print("  RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lin)))


In [ ]:
# Tree model (no need to scale; use raw X for interpretability of importances)
rf = RandomForestRegressor(
    n_estimators=100, max_depth=12, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("RandomForestRegressor")
print("  R2:", r2_score(y_test, y_pred_rf))
print("  MAE:", mean_absolute_error(y_test, y_pred_rf))
print("  RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))


In [ ]:
plt.figure(figsize=(6, 5))
lims = float(y.min()), float(y.max())
plt.scatter(y_test, y_pred_lin, alpha=0.25, s=10, label="Linear")
plt.plot(lims, lims, "k--", lw=1, label="perfect")
plt.xlabel("Actual Billing Amount")
plt.ylabel("Predicted")
plt.legend()
plt.title("LinearRegression: actual vs predicted")
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))
lims = float(y.min()), float(y.max())
plt.scatter(y_test, y_pred_rf, alpha=0.25, s=10, color="darkgreen")
plt.plot(lims, lims, "k--", lw=1)
plt.xlabel("Actual Billing Amount")
plt.ylabel("Predicted")
plt.title("RandomForest: actual vs predicted")
plt.show()


In [ ]:
# Sanity check baseline: always predict train mean (compare R2 to models)
baseline = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
print("Mean baseline R2:", r2_score(y_test, baseline))


In [ ]:
# Example feature importances (Random Forest)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(15)


In [ ]:
# Billing by condition (EDA on original df)
df.groupby("Medical Condition")["Billing Amount"].mean().sort_values(ascending=False)
